# AGMS-Net Phase 2: Regime-Aware Gating (v2 — Scaled Injection)

**Change from Phase 0:** A `RegimeDetector` module extracts 5 statistical features (μ, σ, trend, kurtosis, autocorrelation) from the **raw pre-RevIN input** and injects a learned regime embedding as a **scaled** additive bias into the encoder input before the MSPBlock loop.

**v2 fix:** Added `regime_scale` (learnable scalar, init=0.01) to control injection magnitude. v1 used unconstrained addition which caused MSE-MAE divergence.

**MSPBlock:** Reverted to Phase 0 baseline (static `self.mapping`). Regime injection is at the `Model.encoder()` level.

**Runtime:** GPU required (Runtime > Change runtime type > T4 GPU)

**Dataset:** Same `alibaba.csv` (96 machines, 11206 timesteps)

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Clone Repository (Phase 2 v2 code)

In [ ]:
import os
!rm -rf /content/Workload_Prediction_MTP
!git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('run.py exists:', os.path.exists('run.py'))

# Verify Phase 2 v2 code is present
with open('models/MSCNet.py') as f:
    content = f.read()
    assert 'RegimeDetector' in content, 'ERROR: RegimeDetector not found!'
    assert 'AdaptiveGate' not in content, 'ERROR: AdaptiveGate still present!'
    assert 'regime_scale' in content, 'ERROR: regime_scale not found! Need v2 fix.'
    print('Phase 2 v2 confirmed: RegimeDetector + regime_scale (scaled injection)')
    print('MSPBlock reverted to Phase 0 baseline (confirmed)')

## 3. Install Dependencies

In [ ]:
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

## 4. Upload Dataset

In [ ]:
import os, pandas as pd
%cd /content
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/alibaba.csv'):
    from google.colab import files
    print('Upload alibaba.csv (96 machines, ~11MB):')
    uploaded = files.upload()
    for f in uploaded: os.rename(f, 'dataset/alibaba.csv')
df = pd.read_csv('dataset/alibaba.csv')
n_features = len(df.columns) - 1
print(f'Dataset: {df.shape}, Features: {n_features}')
print(f'Time: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')

## 5. Setup Paths & Symlinks

In [ ]:
import os
%cd /content
DATA_DIR  = '/content/dataset'
WORK_DIR  = '/content/Workload_Prediction_MTP/MSCNet_baseline'
os.makedirs(f'{WORK_DIR}/dataset', exist_ok=True)
if not os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'):
    os.symlink(f'{DATA_DIR}/alibaba.csv', f'{WORK_DIR}/dataset/alibaba.csv')
print('Symlink created:', os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'))
%cd {WORK_DIR}
print('Working directory:', os.getcwd())

## 6. Train All Horizons (H=24, 48, 72, 96)

Each takes ~8-12 minutes on T4. Total ~35-45 min.

In [ ]:
import os, subprocess, sys

HORIZONS = [24, 48, 72, 96]
N_FEATURES = 96
SEQ_LEN = 96

for h in HORIZONS:
    print(f'\n{"="*80}')
    print(f'  Training H={h}')
    print(f'{"="*80}\n')

    cmd = [
        sys.executable, 'run.py',
        '--task_name', 'long_term_forecast',
        '--is_training', '1',
        '--root_path', './dataset/',
        '--data_path', 'alibaba.csv',
        '--model_id', f'alibaba_{SEQ_LEN}_{h}',
        '--model', 'MSCNet',
        '--data', 'custom',
        '--features', 'M',
        '--seq_len', str(SEQ_LEN),
        '--label_len', str(SEQ_LEN),
        '--pred_len', str(h),
        '--e_layers', '3',
        '--d_layers', '1',
        '--factor', '3',
        '--enc_in', str(N_FEATURES),
        '--dec_in', str(N_FEATURES),
        '--c_out', str(N_FEATURES),
        '--d_model', '128',
        '--d_ff', '2048',
        '--n_heads', '4',
        '--des', 'Exp',
        '--itr', '1',
        '--batch_size', '32',
        '--learning_rate', '0.0001',
        '--train_epochs', '100',
        '--patience', '3',
        '--channel', str(N_FEATURES),
        '--seed', '2022',
    ]

    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        print(f'ERROR: H={h} failed with return code {result.returncode}')

## 7. Collect & Compare Results

In [ ]:
import os, re, numpy as np

phase0 = {24: (0.4630, 0.4927), 48: (0.5638, 0.5438), 72: (0.6447, 0.5794), 96: (0.7213, 0.6144)}
paper  = {24: 0.5687, 48: 0.6274, 72: 0.6622, 96: 0.6893}

phase2 = {}
print('--- Raw Results Log ---')
for root, dirs, files in os.walk('results'):
    for f in files:
        if f == 'metrics.npy':
            path = os.path.join(root, f)
            m = np.load(path)
            folder = os.path.basename(root)
            print(f'{folder}')
            print(f'mse:{m[0]}, mae:{m[1]}, rmse:{m[2]}')
            match = re.search(r'alibaba_96_(\d+)_', folder)
            if match:
                h = int(match.group(1))
                phase2[h] = (float(m[0]), float(m[1]))
            print()

print()
print('=' * 105)
print('AGMS-Net Phase 2v2 (RegimeDetector + Scaled) vs Phase 0 Baseline vs Paper (Alibaba, L=96)')
print('=' * 105)
hdr = f'{"H":>4} | {"Phase2 MSE":>10} | {"Phase0 MSE":>10} | {"vs P0":>7} | {"Paper MSE":>10} | {"Phase2 MAE":>10} | {"Phase0 MAE":>10} | {"vs P0":>7}'
print(hdr)
print('-' * 105)
for h in [24, 48, 72, 96]:
    if h in phase2:
        p2_mse, p2_mae = phase2[h]
        p0_mse, p0_mae = phase0[h]
        pap_mse = paper[h]
        d_mse = (p2_mse - p0_mse) / p0_mse * 100
        d_mae = (p2_mae - p0_mae) / p0_mae * 100
        print(f'{h:4d} | {p2_mse:10.4f} | {p0_mse:10.4f} | {d_mse:+6.1f}% | {pap_mse:10.4f} | {p2_mae:10.4f} | {p0_mae:10.4f} | {d_mae:+6.1f}%')
print('=' * 105)

## 8. Regime Embedding & Scale Analysis

Inspect the learned `regime_scale` value and regime detector weights.

In [ ]:
import torch, os, glob

checkpoint_dirs = sorted(glob.glob('checkpoints/long_term_forecast_alibaba_*'))
print(f'Found {len(checkpoint_dirs)} checkpoint(s)')

for ckpt_dir in checkpoint_dirs:
    ckpt_path = os.path.join(ckpt_dir, 'checkpoint.pth')
    if not os.path.exists(ckpt_path):
        continue
    print(f'\n--- {ckpt_path} ---')
    state = torch.load(ckpt_path, map_location='cpu')

    # Check regime_scale (the key new parameter)
    if 'regime_scale' in state:
        print(f'  regime_scale = {state["regime_scale"].item():.6f}  (init was 0.01)')
    else:
        print('  WARNING: regime_scale not found in checkpoint!')

    regime_params = {k: v for k, v in state.items() if 'regime' in k}
    if regime_params:
        print(f'  Regime Detector parameters:')
        for k, v in regime_params.items():
            if k == 'regime_scale':
                continue
            print(f'    {k}: shape={v.shape}, mean={v.mean():.4f}, std={v.std():.4f}')

    total_params = sum(v.numel() for v in state.values())
    regime_param_count = sum(v.numel() for v in regime_params.values())
    print(f'  Total params: {total_params:,}')
    print(f'  Regime params: {regime_param_count:,} ({regime_param_count/total_params*100:.2f}%)')

print('\nRegime parameters look healthy!')

## 9. Model Complexity (MACs & Parameters)

In [ ]:
import torch, sys, os
os.chdir('/content/Workload_Prediction_MTP/MSCNet_baseline')
sys.path.insert(0, '.')

from ptflops import get_model_complexity_info
from types import SimpleNamespace

configs = SimpleNamespace(
    task_name='long_term_forecast', seq_len=96, pred_len=24,
    enc_in=96, dec_in=96, c_out=96, d_model=128, d_ff=2048,
    n_heads=4, e_layers=3, d_layers=1, factor=3, dropout=0.05,
    output_attention=False, activation='gelu', moving_avg=25, channel=96
)

from models.MSCNet import Model
model = Model(configs)
macs, params = get_model_complexity_info(
    model, (96, 96), as_strings=True, print_per_layer_stat=False, verbose=False
)
print(f'MACs: {macs}')
print(f'Parameters: {params}')

## 10. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

horizons = [24, 48, 72, 96]

phase0_mse = [0.4630, 0.5638, 0.6447, 0.7213]
phase0_mae = [0.4927, 0.5438, 0.5794, 0.6144]
paper_mse  = [0.5687, 0.6274, 0.6622, 0.6893]

# Phase 2v2 results - UPDATE THESE after training (from Cell 7 output):
phase2_mse = []  # e.g., [0.4600, 0.5600, 0.6400, 0.7100]
phase2_mae = []  # e.g., [0.4900, 0.5400, 0.5750, 0.6100]

if not phase2_mse:
    print('Phase 2v2 results not yet filled in. Run Cell 7 first, then update this cell.')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(horizons))
    w = 0.25

    ax1.bar(x - w, paper_mse, w, label='Paper (MSCNet)', color='#999999', alpha=0.8)
    ax1.bar(x,     phase0_mse, w, label='Phase 0 (Our Baseline)', color='#4ECDC4', alpha=0.8)
    ax1.bar(x + w, phase2_mse, w, label='Phase 2v2 (Regime Scaled)', color='#FF6B6B', alpha=0.8)
    ax1.set_xlabel('Prediction Horizon (H)')
    ax1.set_ylabel('MSE')
    ax1.set_title('MSE Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(horizons)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    ax2.bar(x - w, phase0_mae, w, label='Phase 0 (Our Baseline)', color='#4ECDC4', alpha=0.8)
    ax2.bar(x,     phase2_mae, w, label='Phase 2v2 (Regime Scaled)', color='#FF6B6B', alpha=0.8)
    ax2.set_xlabel('Prediction Horizon (H)')
    ax2.set_ylabel('MAE')
    ax2.set_title('MAE Comparison')
    ax2.set_xticks(x)
    ax2.set_xticklabels(horizons)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('phase2v2_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: phase2v2_results.png')